# Step 1 Data Background

Dataset ini menyajikan catatan historis terkait pasar alpukat di negara Amerika Serikat, yang sumber data berasal dari Hass Avocado Board. Data ini mencakup pengamatan mingguan dari tahun 2015 hingga awal 2018, yang mendokumentasikan fluktuasi harga rata-rata per unit serta volume penjualan total. Informasi yang tersedia tidak hanya terbatas pada angka penjualan kasar, tetapi juga dirinci berdasarkan kode Price Look-Up (PLU) untuk membedakan ukuran alpukat (Small, Large, dan Extra Large), serta kategori kemasan berupa kantong (bags). Dengan cakupan geografis yang luas, dataset ini memetakan tren di 54 wilayah berbeda.

# Step 2 Import Library & Read Data

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df = pd.read_csv(r"E:\Aplikasi\db_learn\python\avocado.csv")

# Step 3 Analysing Data & Data Validation

In [ ]:
print(f"Total Row : {df.shape[0]}")
print(f"Total Column : {df.shape[1]}\n")
df.info()

In [ ]:
df.describe()

In [ ]:
df.dtypes

In [ ]:
missing_values = df.isnull().sum()
print('Total Missing Value :')
print(missing_values)

In [ ]:
duplicate = df.duplicated().sum
print(f"\nTotal Duplicate Data:{duplicate} ")
print(duplicate)

In [ ]:
df.head()

In [ ]:
for col in df.columns:
    print(f"Kolom '{col}': {df[col].nunique()} nilai unik")

# Step 4 Data Cleansing

In [ ]:
df['Date'] = pd.to_datetime(df['Date'])

In [ ]:
print(df['region'].unique())

In [ ]:
df_clean = df[df['region'] != 'TotalUS']
# Cek jumlah baris yang tersisa
print(f"Jumlah data sebelum filter: {len(df)}")
print(f"Jumlah data setelah filter: {len(df_clean)}")

In [ ]:
print(df_clean['region'].unique())

# Step 5 Data Visualisation

## Tren Harga VS Volume
### Kasus 1
Perhatikan puncak tertinggi garis merah. Di sana harga melonjak drastis hingga hampir 1.85. Lihat garis birunya di bulan yang sama: Garis biru terjun bebas ke titik terendahnya (sekitar 1.75). 
Insight : "pada September 2017, kenaikan harga yang sangat tinggi berkorelasi kuat dengan penurunan drastis volume penjualan".

### Kasus 2
Harga merah turun ke titik rendah (sekitar 1.2). Lihat garis birunya: Garis biru melonjak naik (puncak kedua tertinggi).
Insight: "Saat harga murah, orang membeli alpukat dalam jumlah yang jauh lebih banyak."

### Kesimpulan
Kesimpulan Analisis untuk Laporan Kamu
Secara keseluruhan, dataset Avocado ini menunjukkan Korelasi Negatif yang sangat jelas.

Analisis: Grafik menunjukkan pola "berlawanan arah" (seperti cermin). Hampir di setiap titik di mana garis merah (Price) naik, garis biru (Volume) akan turun. Ini membuktikan bahwa permintaan pasar alpukat sangat sensitif terhadap harga (Price Elasticity). Penjualan paling drastis terjadi saat harga mencapai puncaknya di akhir tahun 2017.

In [ ]:

df_month = df_clean.set_index('Date').resample('ME').agg({
    'AveragePrice': 'mean',
    'Total Volume': 'sum'
})

# 
fig, ax1 = plt.subplots(figsize=(12, 6))

# Garis Pertama: Average Price (Sumbu Y Kiri)
color = 'tab:red'
ax1.set_xlabel('Month')
ax1.set_ylabel('Average Price', color=color)
ax1.plot(df_month.index, df_month['AveragePrice'], color='red', marker='o', label='Price')
ax1.set_ylabel('Average Price', color='red')
ax1.tick_params(axis='y', labelcolor=color)

# Membuat Sumbu Y Kedua (Sumbu Y Kanan)
ax2 = ax1.twinx()
color= 'tab:blue'
ax2.set_ylabel('Total Volume', color=color)
ax2.plot(df_month.index, df_month['Total Volume'], color='blue', marker='o', label='Volume')
ax2.set_ylabel('Total Volume', color='blue')
ax2.tick_params(axis='y', labelcolor=color)

# 1. Menentukan Koordinat Titik Penting (Sept 2017)
# Ganti dengan koordinat yang tepat dari dataset bulanan kamu
x_target =pd.to_datetime('2017-09-30')
y_price = 1.84
y_volume = 175000000

# Lingkaran Harga (Merah)
ax1.scatter(x_target, y_price, s=500, facecolors='none', edgecolors='red', linewidths=2, label='Annotation Price')
# Lingkaran Volume (Biru) - Perhatikan penggunaan ax2
ax2.scatter(x_target, y_volume, s=500, facecolors='none', edgecolors='blue', linewidths=2)

ax1.annotate('peak pricek -> Volume Plummeted',
    xy=(x_target, y_price),
    xytext=(x_target, 1.6),
    arrowprops=dict(facecolor='black', shrink=0.05, width=1, headwidth=8),
    horizontalalignment='center')
    

plt.title('Negative Correlation Between Price and Volume: A Sharp Drop in Sales When Prices Peak ')
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
df.head()

In [ ]:
idxmax_price = df_month['AveragePrice'].idxmax()

y_price = df_month.loc[idxmax_price, 'AveragePrice']
y_volume = df_month.loc[idxmax_price, 'Total Volume']

x_target = idxmax_price
print(x_target)

## Distribusi Harga Per Tipe
Analisis Tren menunjukkan bahwa volume penjualan Alpukat sangat sensitif terhadap kenaikan harga. Analisis Distribusi (Boxplot) ini memperjelas bahwa sensitivitas tersebut kemungkinan besar paling banyak disumbang oleh tipe Konvensional, karena harga mediannya jauh lebih rendah dan rentang harganya lebih stabil. Konsumen tipe Konvensional lebih terbiasa dengan harga murah dan akan langsung bereaksi jika harga naik sedikit saja.

In [ ]:
# Distributin Price Type
plt.figure(figsize=(10,6))
sns.boxplot(x='type', y='AveragePrice', data=df_clean, palette='Set2', hue='type', legend=False)

plt.title('Price Distribution Comparison: Organic vs. Conventional Avocados', fontsize=14)
plt.xlabel('Types of Avocado', fontsize=12)
plt.ylabel('Average Price ($)', fontsize=12)
plt.grid(axis='y', alpha=0.3)
plt.show()

## Correlation Analysis: Determinants of Total Sales
### Temuan Utama
Tujuan dari analisis korelasi ini adalah untuk mengukur seberapa kuat hubungan antara Harga, Total Volume, dan ukuran setiap buah alpukat (Small, Large, XLarge)
1. **Dominasi Ukuran:** "Alpukat yang berukuran Small (4046) dan Large (4225) mempunyai korelasi yang sangat kuat terhadap Total Volume (dengan nilai 0,90 dan 0,87); angka-angka ini menunjukkan bahwa kedua ukuran alpukat tersebut merupakan **penentu utama volume penjualan**".
2. **Hubungan Antara Harga dan Volume:** "Ditemukan adanya korelasi negatif antara harga rata-rata dan volume total (-0,34). Secara matematis, hal ini menunjukkan bahwa kenaikan harga cenderung diikuti oleh penurunan permintaan terhadap alpukat".
3. **Korelasi Antara Ukuran:** "Ukuran Kecil dan Besar menunjukkan korelasi sedang (0,60), yang mengindikasikan bahwa ukuran kedua produk ini sering kali mengikuti **tren permintaan yang sama di pasar**".
### Dampak Bisnis 
#### Rekomendasi Strategis
1. **Prioritas Stok:** Dengan meninjau kembali Korelasi Ukuran 4046 yang sangat Tinggi terhadap Total Volume, memprioritaskan ketersediaan stok  ukuran ini untuk menjaga kestabilan revenue
2. **Sensitivitas Harga:** Untuk harga memiliki korelasi bersifat Negatif, perlu diperhatikan ketika menaikkan harga pada ukuran Small dan Large,  ini dapat berkaitan langsung pada penurunan volume secara signifikan.

In [ ]:
# Select the columns for which you want to view the correlation
cols = ['Total Volume', '4046', '4225', '4770', 'AveragePrice']
corr_matrix = df_clean[cols].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, cmap='RdBu', fmt=' .2f', linewidth=0.5)

plt.title('The Correlation Between Avocado Size and Total Sales', fontsize=14)
plt.show()